# Lab 24-08 · Coupling, Grating Couplers, Bond Parasitics and Retiming

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import siph


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Coupling

*Predict* the coupling between a 4.6 µm fiber mode and a 1.5 µm chip mode.

In [2]:
print(f"closed form {siph.coupling(4.6, 1.5):.6f}, grid overlap {siph.overlap_numeric(4.6, 1.5):.6f}")
d = np.linspace(0, 3, 121)
plt.figure()
for w in (1.5, 3.0, 4.6):
    loss = [-10 * math.log10(siph.coupling(w, w, x)) for x in d]
    plt.plot(d, loss, label=f"matched w = {w} µm")
    print(f"w = {w} µm: 0.5 dB at {w*math.sqrt(0.05*math.log(10)):.2f} µm offset")
plt.xlabel("lateral offset (µm)"); plt.ylabel("coupling loss (dB)"); plt.legend(); plt.grid(True)
plt.show()

closed form 0.347500, grid overlap 0.347500
w = 1.5 µm: 0.5 dB at 0.51 µm offset
w = 3.0 µm: 0.5 dB at 1.02 µm offset
w = 4.6 µm: 0.5 dB at 1.56 µm offset


## Part 2 · Grating coupler

*Predict* the 1-dB bandwidth of Example 24-08.4's grating.

In [3]:
lam = np.linspace(1250, 1370, 24001)
s = siph.grating_spectrum(lam, 1310, 2.8, 10, 4.6)
band = lam[s >= 10 ** -0.1]
print(f"period {siph.grating_period_nm(1310, 2.8, 10):.1f} nm; 1-dB band {band.min():.1f} to {band.max():.1f} nm "
      f"({band.max() - band.min():.1f} nm)")
plt.figure()
plt.plot(lam, 10 * np.log10(s))
for ch in (1271, 1291, 1311, 1331):
    plt.axvline(ch, color="0.6", ls=":")
plt.ylim(-6, 0.5); plt.xlabel("wavelength (nm)"); plt.ylabel("tilt loss (dB)"); plt.grid(True)
plt.show()

period 498.8 nm; 1-dB band 1288.7 to 1331.4 nm (42.7 nm)


## Part 3 · Bond parasitics

*Predict* the wire bond's peaking and both bandwidths.

In [4]:
for name, l in (("wire bond 1 nH", 1e-9), ("hybrid bond 10 pH", 10e-12)):
    f0, z, f3, pk, fr = siph.rlc(50, l, 20e-15)
    print(f"{name}: f0 {f0/1e9:.1f} GHz, zeta {z:.3f}, f3dB {f3/1e9:.1f} GHz, peaking {20*math.log10(pk):.1f} dB")
sim = ngspice("bond.cir")
print(f"ngspice: wire-bond peaking {sim['pkdb1']:.2f} dB, f3dB {sim['f3a']/1e9:.1f} GHz; hybrid f3dB {sim['f3b']/1e9:.0f} GHz")
f = np.logspace(9, 12, 2000)
plt.figure()
for name, l in (("1 nH wire bond", 1e-9), ("10 pH hybrid bond", 10e-12)):
    plt.semilogx(f / 1e9, 20 * np.log10(siph.rlc_mag(f, 50, l, 20e-15)), label=name)
plt.axvline(113.4375 / 2, color="0.5", ls="--", label="Nyquist, 113.4375 GBd")
plt.ylim(-20, 15); plt.xlabel("frequency (GHz)"); plt.ylabel("|H| (dB)"); plt.legend(); plt.grid(True, which="both")
plt.show()

wire bond 1 nH: f0 35.6 GHz, zeta 0.112, f3dB 54.8 GHz, peaking 13.1 dB
hybrid bond 10 pH: f0 355.9 GHz, zeta 1.118, f3dB 195.8 GHz, peaking 0.0 dB
ngspice: wire-bond peaking 13.06 dB, f3dB 54.8 GHz; hybrid f3dB 196 GHz


## Part 4 · Retimed against linear

*Predict* the end-to-end BER of 1 to 4 segments, each good for 10⁻¹² alone.

In [5]:
q0 = 7.0344838253
for n in range(1, 5):
    ql = siph.linear_q([q0] * n)
    print(f"{n} segment(s): retimed {siph.retimed_ber([q0] * n):.1e}, linear Q {ql:.2f}, BER {siph.qfunc(ql):.2e}")

1 segment(s): retimed 1.0e-12, linear Q 7.03, BER 1.00e-12
2 segment(s): retimed 2.0e-12, linear Q 4.97, BER 3.28e-07
3 segment(s): retimed 3.0e-12, linear Q 4.06, BER 2.44e-05
4 segment(s): retimed 4.0e-12, linear Q 3.52, BER 2.18e-04
